In [1]:
# =========================
# STEP 0 — IMPORTS, PATH, FOLDERS
# =========================
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# For display in Jupyter
try:
    from IPython.display import display
except ImportError:
    def display(x): print(x.head() if hasattr(x, "head") else x)

# >>> UPDATE THIS to your local CSV path
DATA_PATH = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Data\raw\diabetic_data.csv"

# === Absolute folders for saving results ===
EDA_DIR     = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\eda_visualizations"
OUTPUTS_DIR = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs"

# Create folders if not exist
os.makedirs(EDA_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("EDA visualizations will be saved to:", EDA_DIR)
print("Outputs will be saved to:", OUTPUTS_DIR)

EDA visualizations will be saved to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\eda_visualizations
Outputs will be saved to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs


In [2]:
# =========================
# STEP 1 — LOAD DATASET
# =========================
df_raw = pd.read_csv(DATA_PATH)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

# (Optional) standardize placeholder missing tokens -> NaN so they don't become fake outliers
PLACEHOLDERS = ["?", "Unknown/Invalid", "Unknown", "None", "N/A", "NA", "NULL", "Not Available"]
df = df_raw.replace(PLACEHOLDERS, np.nan)

print("Total NaNs after standardization:", int(df.isna().sum().sum()))

Raw shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


Total NaNs after standardization: 374020


In [3]:
# =========================
# STEP 2 — NUMERIC COLUMNS & EXCLUSIONS
# =========================
# Exclude IDs/targets from outlier treatment
EXCLUDE_COLS = {"encounter_id", "patient_nbr", "readmitted", "readmitted_30d"}
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in EXCLUDE_COLS]

print("Numeric columns considered (first 20 shown):", numeric_cols[:20], "...")
print("Count:", len(numeric_cols))

Numeric columns considered (first 20 shown): ['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'] ...
Count: 11


In [4]:
# =========================
# STEP 3 — IQR OUTLIER REPORT (BEFORE TREATMENT)
# =========================
def iqr_bounds(s: pd.Series, k: float = 1.5):
    q1 = s.quantile(0.25); q3 = s.quantile(0.75); iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper, q1, q3, iqr

def iqr_outlier_report(frame: pd.DataFrame, cols, k: float = 1.5):
    rows = []
    for col in cols:
        s = frame[col].dropna()
        if s.empty: 
            continue
        lower, upper, q1, q3, iqr = iqr_bounds(s, k)
        n_outliers = int(((frame[col] < lower) | (frame[col] > upper)).sum())
        rows.append({
            "column": col,
            "q1": q1, "q3": q3, "iqr": iqr,
            "lower_bound": lower, "upper_bound": upper,
            "n_outliers": n_outliers
        })
    rep = pd.DataFrame(rows).sort_values("n_outliers", ascending=False)
    return rep

iqr_before = iqr_outlier_report(df, numeric_cols, k=1.5)
display(iqr_before.head(20))

rep_path = os.path.join(OUTPUTS_DIR, "outliers_iqr_report_before.csv")
iqr_before.to_csv(rep_path, index=False)
print("Saved BEFORE IQR report to:", rep_path)

,column,q1,q3,iqr,lower_bound,upper_bound,n_outliers
7,number_outpatient,0.0,0.0,0.0,0.0,0.0,16739
8,number_emergency,0.0,0.0,0.0,0.0,0.0,11383
1,discharge_disposition_id,1.0,4.0,3.0,-3.5,8.5,9818
9,number_inpatient,0.0,1.0,1.0,-1.5,2.5,7049
2,admission_source_id,1.0,7.0,6.0,-8.0,16.0,6956
5,num_procedures,0.0,2.0,2.0,-3.0,5.0,4954
6,num_medications,10.0,20.0,10.0,-5.0,35.0,2557
3,time_in_hospital,2.0,6.0,4.0,-4.0,12.0,2252
0,admission_type_id,1.0,3.0,2.0,-2.0,6.0,341
10,number_diagnoses,6.0,9.0,3.0,1.5,13.5,281


Saved BEFORE IQR report to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs\outliers_iqr_report_before.csv


In [5]:
# =========================
# STEP 4 — CHOOSE METHOD & APPLY OUTLIER TREATMENT
# =========================
# Method options:
#   method = "iqr_remove"   -> remove rows outside [lower, upper]
#   method = "zscore_remove"-> remove rows with |z| > z_thr
#   method = "winsorize"    -> cap values to [lower, upper] (no row removal)
method = "iqr_remove"   # <<< change here if needed

# Parameters
IQR_K   = 1.5   # 1.5 standard; 3.0 stricter
ZS_THR  = 3.0   # z-score threshold
COLS_TO_TREAT = numeric_cols  # or pick a subset like ["time_in_hospital", "num_lab_procedures"]

def apply_outlier_treatment(frame: pd.DataFrame, cols, method="iqr_remove", iqr_k=1.5, z_thr=3.0):
    df2 = frame.copy()

    if method == "iqr_remove":
        # build a row mask and keep only rows within bounds for all treated cols
        mask = pd.Series(True, index=df2.index)
        for col in cols:
            s = df2[col]
            if not pd.api.types.is_numeric_dtype(s): 
                continue
            lb, ub, *_ = iqr_bounds(s.dropna(), iqr_k)
            mask &= (s.isna() | s.between(lb, ub))
        removed = int((~mask).sum())
        df2 = df2[mask].reset_index(drop=True)
        return df2, {"rows_removed": removed, "strategy": f"IQR remove (k={iqr_k})"}

    elif method == "zscore_remove":
        mask = pd.Series(True, index=df2.index)
        for col in cols:
            s = df2[col]
            if not pd.api.types.is_numeric_dtype(s): 
                continue
            mu = s.mean(); sd = s.std(ddof=0)
            if sd == 0 or np.isnan(sd):
                continue
            z = (s - mu) / sd
            mask &= (s.isna() | (z.abs() <= z_thr))
        removed = int((~mask).sum())
        df2 = df2[mask].reset_index(drop=True)
        return df2, {"rows_removed": removed, "strategy": f"Z-score remove (thr={z_thr})"}

    elif method == "winsorize":
        # cap values to bounds (no row removal)
        for col in cols:
            s = df2[col]
            if not pd.api.types.is_numeric_dtype(s): 
                continue
            lb, ub, *_ = iqr_bounds(s.dropna(), iqr_k)
            df2[col] = s.clip(lower=lb, upper=ub)
        return df2, {"rows_removed": 0, "strategy": f"Winsorize (IQR k={iqr_k})"}

    else:
        raise ValueError("Unknown method: choose from 'iqr_remove', 'zscore_remove', 'winsorize'")

print("Applying method:", method)
before_shape = df.shape
df_treated, info = apply_outlier_treatment(df, COLS_TO_TREAT, method=method, iqr_k=IQR_K, z_thr=ZS_THR)
after_shape = df_treated.shape

print("Before:", before_shape, "After:", after_shape)
print("Treatment info:", info)

Applying method: iqr_remove
Before: (101766, 50) After: (56848, 50)
Treatment info: {'rows_removed': 44918, 'strategy': 'IQR remove (k=1.5)'}


In [6]:
# =========================
# STEP 5 — IQR OUTLIER REPORT (AFTER TREATMENT)
# =========================
iqr_after = iqr_outlier_report(df_treated, numeric_cols, k=1.5)
display(iqr_after.head(20))

rep_after_path = os.path.join(OUTPUTS_DIR, "outliers_iqr_report_after.csv")
iqr_after.to_csv(rep_after_path, index=False)
print("Saved AFTER IQR report to:", rep_after_path)

,column,q1,q3,iqr,lower_bound,upper_bound,n_outliers
9,number_inpatient,0.0,0.0,0.0,0.0,0.0,14190
3,time_in_hospital,2.0,5.0,3.0,-2.5,9.5,2702
0,admission_type_id,1.0,2.0,1.0,-0.5,3.5,2111
6,num_medications,10.0,18.0,8.0,-2.0,30.0,1076
1,discharge_disposition_id,1.0,3.0,2.0,-2.0,6.0,457
4,num_lab_procedures,32.0,56.0,24.0,-4.0,92.0,73
2,admission_source_id,1.0,7.0,6.0,-8.0,16.0,0
5,num_procedures,0.0,2.0,2.0,-3.0,5.0,0
7,number_outpatient,0.0,0.0,0.0,0.0,0.0,0
8,number_emergency,0.0,0.0,0.0,0.0,0.0,0


Saved AFTER IQR report to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs\outliers_iqr_report_after.csv


In [7]:
# =========================
# STEP 6 — SAVE CLEANED DATASET
# =========================
# Name the file to reflect the method used
suffix = {
    "iqr_remove": "iqr_removed",
    "zscore_remove": "zscore_removed",
    "winsorize": "winsorized"
}[method]

cleaned_path = os.path.join(OUTPUTS_DIR, f"diabetic_outliers_{suffix}.csv")
df_treated.to_csv(cleaned_path, index=False)
print("Saved cleaned dataset to:", cleaned_path)

Saved cleaned dataset to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs\diabetic_outliers_iqr_removed.csv
